# Data Preprocess 

This script concatenates most recent passive data with backup data and performs basic preprocessing on passive, EMA and Monitoring data.

In [5]:
from pyprojroot import here # define relative paths to the project root (working directory)
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc  
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------

# relative project root
PROJECT_ROOT = here() # '.here' is located as invisible file in the project root working directory
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path

from functions.preprocessing.infer_timeoffset import (
    create_utcday_tzoffset_df,
    merge_fill_tz,
)

# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")        
today_day = pd.Timestamp.today().normalize()       


# --- Path -------------------------------------------------------------

datapath = Path(raw_path) / f"export_tiki_{today_str}"  

## 1. Passive Data

### 1.1 Load most recent passive data

In [6]:
# actual passive + ema_data

# define the pattern for passive data files
file_pattern = os.path.join(datapath, "epoch_part*.csv")

# use glob to find all matching files
file_list = glob.glob(file_pattern)

# sort the file list for consistent ordering
file_list.sort()

# concatenate all CSV files into a single DataFrame
df_complete = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)


In [7]:
# Extract customer identifier and reduce to first 4 characters
df_complete["customer"] = df_complete.customer.str.split("@").str.get(0)
df_complete["customer"] = df_complete["customer"].str[:4]

for col in ["startTimestamp", "endTimestamp"]:
    df_complete[col] = (
        pd.to_datetime(df_complete[col], utc=True, errors="coerce", unit="ms")
    )

#df_complete

In [8]:
df_complete = df_complete[['customer', 'startTimestamp', 'endTimestamp', 'timezoneOffset', 'type',
       'stringValue', 'booleanValue', 'doubleValue', 'longValue']]

### 1.2 Load big backup dataset

In [9]:
# merge with backup data
backup_path = preprocessed_path + "/backup_passive_05052025.parquet"

# backup_path = preprocessed_path + "/backup_passive_last.feather"
df_backup = pd.read_parquet(backup_path)

In [10]:
# make independent copies of both DataFrames to avoid SettingWithCopyWarning (future modifications do not affect any original DataFrame)
df_backup = df_backup.copy()
df_complete = df_complete.copy()

# convert booleanValue to boolean[pyarrow] dtype before concatenation so that it can be saved to .feather later
# alternative to "boolean[pyarrow]" is "boolean", but it is experimental and may change in future pandas versions
df_backup['booleanValue'] = df_backup['booleanValue'].astype('boolean[pyarrow]')
df_complete['booleanValue'] = df_complete['booleanValue'].astype('boolean[pyarrow]')

In [11]:
# latest timestamp from the backup dataset
latest_timestamp = df_backup['startTimestamp'].max()

# filter from df_complete only those entries that are newer than what’s already in the backup
df_complete_filtered = df_complete[df_complete['startTimestamp'] > latest_timestamp]

### 1.3 Concat Backup and most recent data

In [12]:
# update the backup by concatenating only the newly filtered rows from df_complete, creating an up-to-date backup
df_backup_recent = pd.concat([df_backup, df_complete_filtered], ignore_index=True)

### 1.4 Rename variable names and create additional columns 

In [13]:
# define a clear mapping for rename columns
rename_columns = {"customer": "id",
                  "type": "modality",
                  "startTimestamp": "timestamp_start",
                  "endTimestamp": "timestamp_end",
                  "booleanValue": "boolean_value",
                  "doubleValue": "double_value",
                  "longValue": "long_value",
                  "timezoneOffset": "timezone_offset"}

# apply renaming
df_backup_recent = df_backup_recent.rename(columns=rename_columns)

In [14]:
# create a unified float_value column:
# use 'doubleValue' where available (more precise), otherwise use 'longValue'
df_backup_recent['float_value'] = df_backup_recent['double_value'].fillna(df_backup_recent['long_value'])


In [15]:
# drop original value columns that have been unified into 'float_value' + 'stringValue' (because only ECG data are stored as string for period March - November 2023) + 'createdtAt'
df_backup_recent = df_backup_recent.drop(columns=['double_value', 'long_value', 'stringValue'])


In [16]:
# create a time_interval (duration in seconds) column
df_backup_recent['time_interval'] = (
    df_backup_recent['timestamp_end'] - df_backup_recent['timestamp_start']
).dt.total_seconds()

# create a start_date and start_hour column
df_backup_recent['start_date']  = df_backup_recent['timestamp_start'].dt.normalize()
df_backup_recent['start_hour'] = df_backup_recent['timestamp_start'].dt.hour

### 1.5 Infer Timezone Offset

In [17]:
df_tz = create_utcday_tzoffset_df(df_backup_recent)


2026-03-23 22:15:09,800 [INFO] [GPS] Could not infer tz for id NGlo at 2023-11-06 00:00:00+00:00 with potential tzs [  60. -300.    0.] - gps_multiple_conflict_with_previous_no_next
2026-03-23 22:15:12,932 [INFO] [ActivityDetailCreatedAt] Could not infer tz for id S5sH at 2024-07-16 00:00:00+00:00 with potential tzs [420. 180.] - activitydetail_multiple_conflict_with_both
2026-03-23 22:15:36,056 [INFO] [Interpolate] Could not infer tz for id 3oNs at 2024-07-08 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-23 22:15:36,073 [INFO] [Interpolate] Could not infer tz for id 3oNs at 2024-07-09 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-23 22:16:43,055 [INFO] [Interpolate] Could not infer tz for id 9nSQ at 2024-04-12 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-23 22:17:17,223 [INFO] [Interpolate] Could not infer tz for id BpVE at 2025-10-01 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-23 22:18:22,607 [INFO] [Interpolate] Could not infer tz for i

In [18]:
df_tz["inferred_tzoffset_timedelta"] = pd.to_timedelta(
    df_tz["inferred_tzoffset"], unit="min"
)

In [19]:
df_tz.columns

Index(['id', 'day', 'inferred_tzoffset', 'inferred_tzoffset_source',
       'inferred_tzoffset_timedelta'],
      dtype='object')

In [20]:
df_backup_recent = df_backup_recent.merge(
    df_tz,
    left_on=["id", "start_date"],
    right_on=["id", "day"],
    how="left",
)
#df_backup_recent.drop(columns=["day"], inplace=True)  # remove day from df_tz

In [21]:
df_backup_recent["local_timestamp_start"] = (
    df_backup_recent["timestamp_start"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent["local_timestamp_end"] = (
    df_backup_recent["timestamp_end"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent.head()

,id,timestamp_start,timestamp_end,timezone_offset,modality,boolean_value,createdAt,float_value,time_interval,start_date,start_hour,day,inferred_tzoffset,inferred_tzoffset_source,inferred_tzoffset_timedelta,local_timestamp_start,local_timestamp_end
0,7yK3,2023-03-14 14:21:59+00:00,NaT,120.0,Latitude,<NA>,NaT,-53.650000,NaN,2023-03-14 00:00:00+00:00,14,2023-03-14 00:00:00+00:00,120.0,gps_single,0 days 02:00:00,2023-03-14 16:21:59,NaT
1,7yK3,2023-03-14 14:21:59+00:00,NaT,120.0,Longitude,<NA>,NaT,-58.801466,NaN,2023-03-14 00:00:00+00:00,14,2023-03-14 00:00:00+00:00,120.0,gps_single,0 days 02:00:00,2023-03-14 16:21:59,NaT
2,7yK3,2023-03-14 14:22:38+00:00,NaT,120.0,Latitude,<NA>,NaT,-53.650010,NaN,2023-03-14 00:00:00+00:00,14,2023-03-14 00:00:00+00:00,120.0,gps_single,0 days 02:00:00,2023-03-14 16:22:38,NaT
3,7yK3,2023-03-14 14:22:38+00:00,NaT,120.0,Longitude,<NA>,NaT,-58.801466,NaN,2023-03-14 00:00:00+00:00,14,2023-03-14 00:00:00+00:00,120.0,gps_single,0 days 02:00:00,2023-03-14 16:22:38,NaT
4,7yK3,2023-03-14 14:23:38+00:00,NaT,120.0,Latitude,<NA>,NaT,-53.650000,NaN,2023-03-14 00:00:00+00:00,14,2023-03-14 00:00:00+00:00,120.0,gps_single,0 days 02:00:00,2023-03-14 16:23:38,NaT


In [22]:
assert df_backup_recent.inferred_tzoffset.isna().sum() == 0, (
    "There are missing inferred timezone offsets!"
)

In [23]:
# just to make sure that we don't use them anymore later
del df_complete_filtered
del df_complete

Next:

1. Since we want to include 'for_id' and 'study_version' in our `passive_data` data frame, we need to extract these data from the monitoring sheet. This is done in section 2

2. Additionally the `monitoring_data` data frame is set up in section 2


## 2. Monitoring data

In [24]:
# import data
df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")

In [25]:
# get an overview of the monitoring data
#df_monitoring.head()

In [26]:
# show columns of monitoring data
print(df_monitoring.columns.tolist())

['FOR_ID', 'EMA_ID', 'Pseudonym', 'Studienversion', 'Status', 'Besonderes', 'T20=Post', 'Start EMA Baseline', 'Ende EMA Baseline', 'Terminpräferenz', 'Termin 1. Gespräch', 'Wer führt 1. Gespräch?', 'Telefonat stattgefunden?', 'Baseline T20 Update verschickt?', 'Freischaltung/ Start EMA T20', 'Ende EMA T20', 'Freischaltung/ Start EMA Post', 'Ende EMA Post', 'Post T20 Update verschickt?', 'Studienende/ Dropout Mail verschickt?', 'Status reduziert', 'Dropout vor KV', 'Dropout nach KV', 'Dropout Grund (frei)', 'Zeitpunkt SP6 Dropout (last day passive/gps)']


In [27]:
df_monitoring = df_monitoring.copy()

df_monitoring.rename(columns = {"Pseudonym": "id",
                                "FOR_ID": "for_id",
                                "EMA_ID": "ema_id", 
                                "Status": "study_status",
                                "Studienversion":"study_version", 
                                "Start EMA Baseline": "ema_base_start", 
                                "Ende EMA Baseline": "ema_base_end", 
                                "Freischaltung/ Start EMA T20": "ema_t20_start",
                                "Ende EMA T20":"ema_t20_end", 
                                "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", 
                                "T20=Post":"t20_post" }, 
                     inplace=True)

df_monitoring = df_monitoring[['for_id', 'ema_id', 'id', 'study_version', 'study_status',
       't20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end']]

df_monitoring["id"] = df_monitoring["id"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)


### 2.1 Merge relevant columns with passive data

In [28]:
df_monitoring_short = df_monitoring[["id", "for_id","study_version"]]

#### 2.2 Final `passive_data` data frame

In [29]:
df_backup_recent= df_backup_recent.merge(df_monitoring_short, on="id", how="right")

In [30]:
df_backup_recent.head()

,id,timestamp_start,timestamp_end,timezone_offset,modality,boolean_value,createdAt,float_value,time_interval,start_date,start_hour,day,inferred_tzoffset,inferred_tzoffset_source,inferred_tzoffset_timedelta,local_timestamp_start,local_timestamp_end,for_id,study_version
0,4MLe,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,120.0,Steps,<NA>,NaT,6.00,60.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,0 days 01:00:00,2023-05-17 15:44:00,2023-05-17 15:45:00,FOR11905,Lang
1,4MLe,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,120.0,ActiveBurnedCalories,<NA>,NaT,0.14,60.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,0 days 01:00:00,2023-05-17 15:44:00,2023-05-17 15:45:00,FOR11905,Lang
2,4MLe,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,120.0,CoveredDistance,<NA>,NaT,4.62,60.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,0 days 01:00:00,2023-05-17 15:44:00,2023-05-17 15:45:00,FOR11905,Lang
3,4MLe,2023-05-17 14:58:01+00:00,2023-05-17 14:58:38+00:00,120.0,HeartRate,<NA>,NaT,74.00,37.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,0 days 01:00:00,2023-05-17 15:58:01,2023-05-17 15:58:38,FOR11905,Lang
4,4MLe,2023-05-17 18:00:55+00:00,NaT,120.0,HeartRate,<NA>,NaT,95.00,NaN,2023-05-17 00:00:00+00:00,18.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,0 days 01:00:00,2023-05-17 19:00:55,NaT,FOR11905,Lang


In [31]:
# ensure data types are coded correctly
df_backup_recent['boolean_value'] = df_backup_recent['boolean_value'].astype('boolean[pyarrow]')
df_backup_recent['study_version'] = df_backup_recent['study_version'].astype('string')
df_backup_recent['modality'] = df_backup_recent['modality'].astype('string')
df_backup_recent['id'] = df_backup_recent['id'].astype('category')

In [32]:
# Get a list of columns to drop (all columns not in keep_cols)
keep_cols_passive = ['id','for_id', 'modality', 'timestamp_start','timestamp_end',
    'local_timestamp_start', 'local_timestamp_end','time_interval', 'float_value', 'boolean_value','start_date', 
    'start_hour', "timezone_offset", 'study_version']

# final passive data frame
df_passive_final = df_backup_recent[keep_cols_passive]

#### 2.3 Final `monitoring_data` data frame

In [33]:
#??

## 3. EMA data

#### 3.1 Load, match and rename relevant data from separate .csv files

In [34]:

# Beispiel: datapath = Path("/pfad/zum/verzeichnis")
session        = pd.read_csv(datapath / "questionnaireSession.csv", low_memory=False)
answers        = pd.read_csv(datapath / "answers.csv", low_memory=False)
choice         = pd.read_csv(datapath / "choice.csv", low_memory=False)
questions      = pd.read_csv(datapath / "questions.csv", low_memory=False)
questionnaire  = pd.read_csv(datapath / "questionnaires.csv", low_memory=False)  # ohne Komma!


In [35]:
# session data
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 
session["user"] = session["user"].str[:4]
session.rename(columns = {"id":"session_id",
                          "user":"id",
                          "completedAt": "timestamp_beep_completion", 
                          "createdAt": "timestamp_item_completion", 
                          "expirationTimestamp": "timestamp_beep_expiration",
                          "sessionRun":"beep_num_run",
                          "study":"schedule_chronotype"}, inplace=True)
session['measurement_burst'] = session['schedule_chronotype'].map(study_mapping)
session['schedule_chronotype'] = session['schedule_chronotype'].map(chronotype_mapping)
# Parse epoch‑ms columns as UTC and drop tz info -> naive UTC
for col in ["timestamp_item_completion", "timestamp_beep_completion", "timestamp_beep_expiration"]:
    session[col] = (
        pd.to_datetime(session[col], unit="ms", utc=True, errors="coerce")
    )

df_sess = session[["id","session_id", "measurement_burst", "beep_num_run", "timestamp_item_completion", "timestamp_beep_completion", "schedule_chronotype", "timestamp_beep_expiration"]]

In [36]:
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 

answers["user"] = answers["user"].str[:4]
answers = answers[["user", "questionnaire", "study", "question", "element",
                   "createdAt", "order", "questionnaireSession"]]

answers["createdAt"] = (
    pd.to_datetime(answers["createdAt"], unit="ms", utc=True, errors="coerce")
)
answers['measurement_burst'] = answers['study'].map(study_mapping)
answers['schedule_chronotype'] = answers['study'].map(chronotype_mapping)

answers.rename(columns={"user": "id", 
                        "createdAt": "timestamp_item_completion",
                        "questionnaire": "beep_type",
                        "question":"item_code_map",
                        "order":"item_order",
                        "questionnaireSession":"session_id",
                        }, inplace=True)
answers.drop(columns=["study"], inplace=True)

In [37]:
# item description data
choice = choice[["element", "choice_id", "text", "question"]]
choice.rename(columns={"text":"response_text",
                       "choice_id":"response", 
                       "question":"item_code_map"}, inplace=True)

In [38]:
# question description data
questions = questions[["id", "title"]]
questions.rename(columns={"id":"item_code_map",
                          "title":"item"}, inplace=True)

In [39]:
questionnaire = questionnaire[["id", "name"]]
questionnaire.rename(columns={"id":"beep_type",
                              "name":"beep_type_name"}, inplace=True)

In [40]:
answer_merged = pd.merge(answers, choice, on= ["item_code_map","element"])
answer_merged = pd.merge(answer_merged, questions, on= "item_code_map")
answer_merged = pd.merge(answer_merged, questionnaire, on= "beep_type")
answer_merged["date"] = answer_merged["timestamp_item_completion"].dt.normalize()

#### 3.2 Calculate auxiliary variables

In [41]:
df_monitoring_ema = df_monitoring[["id", "for_id","study_version", 'ema_base_start', 'ema_base_end',
       'ema_t20_start', 'ema_t20_end', 'ema_post_start', 'ema_post_end', 't20_post']]

In [42]:
bursts = [("base", 0), ("t20", 1), ("post", 2)]

out = []
for burst_name, burst_code in bursts:
    tmp = df_monitoring_ema[[
        "id", "for_id", "study_version", "t20_post",
        f"ema_{burst_name}_start", f"ema_{burst_name}_end"
    ]].copy()

    tmp = tmp.rename(columns={
        f"ema_{burst_name}_start": "ema_burst_start",
        f"ema_{burst_name}_end":   "ema_burst_end",
    })
    tmp["measurement_burst"] = burst_code
    out.append(tmp)

df_monitoring_ema_long = (
    pd.concat(out, ignore_index=True)
      # optional: drop rows where the burst is entirely missing
      .dropna(subset=["ema_burst_start", "ema_burst_end"], how="all")
      .sort_values(["id", "measurement_burst"])
      .reset_index(drop=True)
)


In [43]:
answer_merged = pd.merge(answer_merged, df_monitoring_ema_long, on = ["id", "measurement_burst"])

In [44]:
df_ema_content = answer_merged.copy()

In [45]:
meta_cols = ['id','for_id','date','response_text','item_code_map','beep_type' ,'beep_type_name',
              'element', 'item_order', 'session_id']
df_ema_meta = df_ema_content[meta_cols].copy()

In [46]:
df_ema_content = df_ema_content.drop(columns=['response_text','item_code_map','beep_type' ,'beep_type_name',
              'element', 'item_order']) 

In [47]:
df_sess_short = df_sess[["id", "session_id",  "beep_num_run",'timestamp_beep_completion']].copy()

In [48]:
df_ema_meta = df_ema_meta.merge(df_sess_short, on=["id", "session_id"], how="left")

#### 3.3 Calculate EMA coverage

In [49]:
df_sess_short_compliance = df_sess[["id", "session_id", 'timestamp_beep_completion']].copy()

In [50]:
df_ema_content = df_ema_content.merge(df_sess_short_compliance, on=["id", "session_id"], how="left")

In [51]:
df_ema_content["n_beeps_beeps_completed_per_burst"] = (
    df_ema_content
    .groupby(["measurement_burst", "id"])["timestamp_beep_completion"]
    .transform("nunique"))

#### 3.3 Calculate auxiliary variables

In [52]:
df_ema_content = answer_merged.copy()

In [53]:
# 1. Date and Time Manipulations
df_ema_content['weekday'] = df_ema_content['timestamp_item_completion'].dt.day_name()


# 1a. Season
def get_season(month):
    if month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    elif month in [9, 10, 11]:
        return 3
    else:
        return 4

df_ema_content['season'] = df_ema_content['timestamp_item_completion'].dt.month.apply(get_season)

# 1b. Time of Day
def get_time_of_day(hour):
    if 5 <= hour < 8:
        return 1
    elif 8 <= hour < 12:
        return 2
    elif 12 <= hour < 17:
        return 3
    elif 17 <= hour < 21:
        return 4
    else:
        return 5

df_ema_content['time_of_day'] = df_ema_content['timestamp_item_completion'].dt.hour.apply(get_time_of_day)
df_ema_content['item'] = df_ema_content['item'].str.replace('_morning', '', regex=False)

# 3. Weekend Indicator
df_ema_content['weekend'] = df_ema_content['weekday'].isin(['Saturday', 'Sunday']).astype(int)

# 4. Questionnaire Number
df_ema_content['nr_beep_daily'] = df_ema_content['beep_type_name'].str.extract(r'(\d+)').astype(float)

# 5. Count unique questionnaires per day
df_ema_content['n_beeps_completed_per_day'] = (
    df_ema_content.groupby(['measurement_burst', 'id', 'date'])['beep_type_name']
                  .transform('nunique')
)

# 6. Unique Day Identifier
df_ema_content['quest_nr_str'] = df_ema_content['nr_beep_daily'].fillna('unknown').astype(str)
df_ema_content['beep_per_person_id'] = (
    df_ema_content['date'].dt.strftime('%Y%m%d') + '_' + df_ema_content['quest_nr_str']
)

In [ ]:
"""

# 7. (ersetzt) Relative Start/End pro Phase & Customer
df_ema_content['ema_relative_start'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('min')
)
df_ema_content['ema_relative_end'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('max')
)

# 8. Absolute & Relative Day Index
df_ema_content['absolute_day_index'] = (
    df_ema_content['date'] - df_ema_content['ema_relative_start']
).dt.days + 1

df_ema_content['relative_day_index'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date']
                  .rank(method='dense').astype(int)
)

"""

In [ ]:
"""

# 9. Filter absolute_day_index > 16
max_allowed_days = 16
#df_ema_content = df_ema_content[df_ema_content['absolute_day_index'] <= max_allowed_days].reset_index(drop=True)

# 10. Check
high_indices = df_ema_content[df_ema_content['absolute_day_index'] > max_allowed_days]
high_indices_id = high_indices.id.unique().tolist()
if not high_indices.empty:
    print("Warning: High absolute_day_index vorhanden:", high_indices['id'].unique())
else:
    print("All entries have absolute_day_index <= 16.")

"""

 '9UHX' 'eFL8' 'uxmL' 'hcgv' '5kw5' '2gei' 'p1zH' 'rHcL' 'tdYZ' 'mASG']


In [ ]:
"""

df_max_day = (
    df_ema_content.loc[
        df_ema_content["id"].isin(high_indices_id) & (df_ema_content["absolute_day_index"] > 16),
        ["id", "measurement_burst", "ema_relative_start", "ema_relative_end", "absolute_day_index", "beep_per_person_id"]
    ]
    .groupby(["id", "measurement_burst", "ema_relative_start", "ema_relative_end"], as_index=False)
    .agg(
        max_absolute_day_index=("absolute_day_index", "max"),
        n_unique_beeps_after_day16=("beep_per_person_id", "nunique"),
    )
    .sort_values(["id", "measurement_burst", "max_absolute_day_index"])
)

df_max_day_burst0 = df_max_day.loc[df_max_day["measurement_burst"] == 0]
df_max_day_burst1 = df_max_day.loc[df_max_day["measurement_burst"] == 1]
df_max_day_burst2 = df_max_day.loc[df_max_day["measurement_burst"] == 2]

df_max_day_burst0

"""

,id,measurement_burst,ema_relative_start,ema_relative_end,max_absolute_day_index,n_unique_beeps_after_day16
3,5qL5,0,2023-06-12 00:00:00+00:00,2023-07-02 00:00:00+00:00,21,1
6,BdSf,0,2023-07-11 00:00:00+00:00,2023-07-31 00:00:00+00:00,21,1
7,GjiG,0,2023-08-08 00:00:00+00:00,2023-08-28 00:00:00+00:00,21,1
9,M72F,0,2023-08-29 00:00:00+00:00,2023-09-18 00:00:00+00:00,21,1
10,UfMn,0,2024-01-25 00:00:00+00:00,2024-07-29 00:00:00+00:00,187,1
17,nrqZ,0,2023-08-02 00:00:00+00:00,2023-08-22 00:00:00+00:00,21,1
19,p4A1,0,2023-09-15 00:00:00+00:00,2023-10-05 00:00:00+00:00,21,1


In [53]:
df_max_day_burst1

,id,measurement_burst,ema_relative_start,ema_relative_end,max_absolute_day_index,n_unique_beeps_after_day16
1,5kw5,1,2024-07-03 00:00:00+00:00,2025-03-13 00:00:00+00:00,254,20
4,9Ok2,1,2024-10-04 00:00:00+00:00,2024-10-26 00:00:00+00:00,23,46
5,9UHX,1,2024-10-23 00:00:00+00:00,2025-04-30 00:00:00+00:00,190,208
8,LQFF,1,2024-07-31 00:00:00+00:00,2024-08-21 00:00:00+00:00,22,40
11,eFL8,1,2024-09-25 00:00:00+00:00,2024-12-05 00:00:00+00:00,72,22
12,fx8B,1,2024-07-14 00:00:00+00:00,2024-11-04 00:00:00+00:00,114,31
14,hcgv,1,2024-10-31 00:00:00+00:00,2025-03-03 00:00:00+00:00,124,62
18,p1zH,1,2024-07-21 00:00:00+00:00,2025-04-17 00:00:00+00:00,271,48
21,tdYZ,1,2024-12-30 00:00:00+00:00,2025-11-03 00:00:00+00:00,309,11
22,uxmL,1,2024-09-10 00:00:00+00:00,2025-01-25 00:00:00+00:00,138,109


In [ ]:
df_max_day_burst2

In [ ]:
# Remove Beeps after Day 16 for the Measurement Burst 0
df_ema_content = df_ema_content.loc[
    ~(
        (df_ema_content["measurement_burst"] == 0) &
        (df_ema_content["absolute_day_index"] > 16)
    )
].copy()

Procedure to clean repeated activations at T20 and TPost:
1. detect repeated runs
2. assign an run_id within burst
3. choose the valid run
4. compute start/end indices within that chosen run 

In [ ]:
# detect repeated runs within each burst (this creates a new run whenever the gap between consecutive days is larger than the threshold)

# sort first
df_ema_content = df_ema_content.sort_values(['id', 'measurement_burst', 'date']).copy()

# previous date within participant and burst
df_ema_content['prev_date'] = df_ema_content.groupby(
    ['id', 'measurement_burst']
)['date'].shift()

# gap in days 
df_ema_content['gap_days'] = (
    df_ema_content['date'] - df_ema_content['prev_date']
).dt.days

# threshold (definition when a new activation starts) 
gap_threshold = 2

df_ema_content['new_activation'] = (
    df_ema_content['gap_days'].isna() |
    (df_ema_content['gap_days'] > gap_threshold)
)

# run_id
df_ema_content['run_id'] = df_ema_content.groupby(
    ['id', 'measurement_burst']
)['new_activation'].cumsum()

df_ema_content.head()

,id,beep_type,item_code_map,element,timestamp_item_completion,item_order,session_id,measurement_burst,schedule_chronotype,response,...,time_of_day,weekend,nr_beep_daily,n_beeps_completed_per_day,quest_nr_str,beep_per_person_id,prev_date,gap_days,new_activation,run_id
137409,05kz,68,327,1875.0,2023-10-10 10:07:04.089000+00:00,0.0,16263,0,1,3,...,2,0,2.0,5,2.0,20231010_2.0,NaT,NaN,True,1
137411,05kz,68,319,1738.0,2023-10-10 10:07:07.631000+00:00,0.0,16263,0,1,4,...,2,0,2.0,5,2.0,20231010_2.0,2023-10-10 00:00:00+00:00,0.0,False,1
137416,05kz,68,318,1730.0,2023-10-10 10:07:12.651000+00:00,0.0,16263,0,1,3,...,2,0,2.0,5,2.0,20231010_2.0,2023-10-10 00:00:00+00:00,0.0,False,1
137418,05kz,68,323,1852.0,2023-10-10 10:07:15.863000+00:00,0.0,16263,0,1,4,...,2,0,2.0,5,2.0,20231010_2.0,2023-10-10 00:00:00+00:00,0.0,False,1
137422,05kz,68,330,1893.0,2023-10-10 10:07:20.771000+00:00,0.0,16263,0,1,3,...,2,0,2.0,5,2.0,20231010_2.0,2023-10-10 00:00:00+00:00,0.0,False,1


In [ ]:
# Quality Check: check who has reapeted runs (optional, but useful for quality check)
run_counts = (
    df_ema_content.groupby(['id', 'measurement_burst'])['run_id']
    .nunique()
    .reset_index(name='n_runs')
)

repeated_runs = run_counts[run_counts['n_runs'] > 1]

print("Participants/bursts with repeated runs:")
print(repeated_runs)

Participants/bursts with repeated runs:
       id  measurement_burst  n_runs
6    0RA8                  0       3
18   1rfO                  0       2
19   1rfO                  1       3
20   1udr                  0       3
21   1udr                  1       2
..    ...                ...     ...
695  wGQ8                  2       3
719  y310                  0       2
721  yFws                  1       3
722  yFws                  2       2
733  ywR8                  0       2

[119 rows x 3 columns]


In [ ]:
# summarize runs (helps to choose the valid run)
run_summary = (
    df_ema_content.groupby(['id', 'measurement_burst', 'run_id'], as_index=False)
    .agg(
        run_start=('date', 'min'),
        run_end=('date', 'max'),
        n_days=('date', 'nunique'),
        n_records=('date', 'size'),
        n_beeps=('beep_per_person_id', 'nunique')
    )
)

#run_summary.head()

,id,measurement_burst,run_id,run_start,run_end,n_days,n_records,n_beeps
0,05kz,0,1,2023-10-10 00:00:00+00:00,2023-10-25 00:00:00+00:00,16,1642,49
1,08d6,0,1,2024-09-04 00:00:00+00:00,2024-09-18 00:00:00+00:00,15,2867,88
2,0Aly,0,1,2024-04-08 00:00:00+00:00,2024-04-22 00:00:00+00:00,13,1421,43
3,0Aly,1,1,2025-01-17 00:00:00+00:00,2025-01-31 00:00:00+00:00,14,1534,46
4,0Aly,2,1,2025-08-27 00:00:00+00:00,2025-09-11 00:00:00+00:00,16,1809,54


In [59]:
# choose the valid run per participant and burst (= the one with the most days and beeps)
selected_runs = (
    run_summary.sort_values(
        ['id', 'measurement_burst', 'n_days', 'n_beeps', 'run_id'],
        ascending=[True, True, False, False, False]
    )
    .drop_duplicates(['id', 'measurement_burst'])
    [['id', 'measurement_burst', 'run_id']]
)

# keep only selected run
df_ema_content = df_ema_content.merge(
    selected_runs,
    on=['id', 'measurement_burst', 'run_id'],
    how='inner'
)

In [64]:
# 7. relative start/end per selected run
df_ema_content['ema_relative_start'] = (
    df_ema_content.groupby(['id', 'measurement_burst', 'run_id'])['date'].transform('min')
)

df_ema_content['ema_relative_end'] = (
    df_ema_content.groupby(['id', 'measurement_burst', 'run_id'])['date'].transform('max')
)

# 8. absolute & relative day index
df_ema_content['absolute_day_index'] = (
    df_ema_content['date'] - df_ema_content['ema_relative_start']
).dt.days + 1

df_ema_content['relative_day_index'] = (
    df_ema_content.groupby(['id', 'measurement_burst', 'run_id'])['date']
    .rank(method='dense')
    .astype(int)
)

# 9. check day index > 16
max_allowed_days = 16

high_indices = df_ema_content[df_ema_content['absolute_day_index'] > max_allowed_days]
high_indices_id = high_indices.id.unique().tolist()

if not high_indices.empty:
    print("Warning: High absolute_day_index vorhanden:", high_indices['id'].unique())
else:
    print("All entries have absolute_day_index <= 16.")


In [65]:
# 10. summary of problematic cases
df_max_day = (
    df_ema_content.loc[
        df_ema_content["id"].isin(high_indices_id) & (df_ema_content["absolute_day_index"] > 16),
        ["id", "measurement_burst", "run_id", "ema_relative_start", "ema_relative_end", "absolute_day_index", "beep_per_person_id"]
    ]
    .groupby(["id", "measurement_burst", "run_id", "ema_relative_start", "ema_relative_end"], as_index=False)
    .agg(
        max_absolute_day_index=("absolute_day_index", "max"),
        n_unique_beeps_after_day16=("beep_per_person_id", "nunique"),
    )
    .sort_values(["id", "measurement_burst", "run_id", "max_absolute_day_index"])
)

df_max_day_burst0 = df_max_day.loc[df_max_day["measurement_burst"] == 0]
df_max_day_burst1 = df_max_day.loc[df_max_day["measurement_burst"] == 1]
df_max_day_burst2 = df_max_day.loc[df_max_day["measurement_burst"] == 2]


In [69]:
df_max_day_burst1.head()

,id,measurement_burst,run_id,ema_relative_start,ema_relative_end,max_absolute_day_index,n_unique_beeps_after_day16
0,9UHX,1,1,2024-10-23 00:00:00+00:00,2024-11-15 00:00:00+00:00,24,42
1,LQFF,1,1,2024-07-31 00:00:00+00:00,2024-08-21 00:00:00+00:00,22,40


In [ ]:
# final filter
df_ema_content = df_ema_content.loc[
    df_ema_content["absolute_day_index"] <= max_allowed_days
].copy()

In [ ]:
# Remove out of phase data for Measurement Burst 1 and 2 (self-activation was possible, leading to false-activation of study phases)


In [ ]:
# 11. Questionnaire Counter
df_unique = df_ema_content.drop_duplicates(subset=['id', 'measurement_burst', 'beep_per_person_id']).copy()
df_unique['relative_beep_counter'] = df_unique.groupby(['id', 'measurement_burst']).cumcount() + 1
df_ema_content = df_ema_content.merge(
    df_unique[['id', 'measurement_burst', 'beep_per_person_id', 'relative_beep_counter']],
    on=['id', 'measurement_burst', 'beep_per_person_id'],
    how='left'
)

# 12. Missing Data behandeln
df_ema_content['measurement_burst'] = df_ema_content['measurement_burst'].fillna('unknown')
df_ema_content['absolute_day_index'] = df_ema_content['absolute_day_index'].where(
    df_ema_content['ema_relative_start'].notna(), np.nan
)

### 3.4 merge the inferred tz offsets

In [ ]:
# uncomment if want to run this cell multiple times
# if "infered_tzoffset" in df_sess.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_sess")
#     df_sess = df_sess.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])
# if "inferred_tzoffset" in df_ema_content.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_ema_content")
#     df_ema_content = df_ema_content.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])

df_ema_meta = merge_fill_tz(
    df_ema_meta, df_tz, day_col="date", customer_col="id"
)
df_ema_content = merge_fill_tz(
    df_ema_content, df_tz, day_col="date", customer_col="id"
)

In [ ]:
df_ema_content.drop(columns=['element', 'item_order', 'session_id'], inplace=True) 

### Export passive, EMA and Monitoring

In [ ]:
backup_path = raw_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(backup_path)

preprocessed_path_final = preprocessed_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(preprocessed_path_final)

#preprocessed_path_freezed_final = preprocessed_path_freezed + "/code_check" + "/backup_passive_recent.parquet"
#df_passive_final.to_parquet(preprocessed_path_freezed_final)

with open(preprocessed_path + '/ema_meta.pkl', 'wb') as file:
    pickle.dump(df_ema_meta, file)

    
with open(preprocessed_path + '/monitoring_data.pkl', 'wb') as file:
    pickle.dump(df_monitoring, file)

    
with open(preprocessed_path + '/ema_content.pkl', 'wb') as file:
    pickle.dump(df_ema_content, file)

In [ ]:

# Export ema meta as CSV
df_ema_path = preprocessed_path + '/ema_meta.csv'
df_ema_meta.to_csv(df_ema_path, index=False)

# Export df_monitoring as CSV
df_monitoring_csv_path = preprocessed_path + '/monitoring_data.csv'
df_monitoring.to_csv(df_monitoring_csv_path, index=False)

# Export df_ema_content as CSV
df_ema_content_csv_path = preprocessed_path + '/ema_content.csv'
df_ema_content.to_csv(df_ema_content_csv_path, index=False)

# Export df_ema_content as CSV to freezed for data check
#df_ema_content_csv_path = preprocessed_path_freezed +'/code_check' +'/ema_content_recent.csv'
#df_ema_content.to_csv(df_ema_content_csv_path, index=False)

